<a href="https://colab.research.google.com/github/himavarshinirekhapalli-jpg/Hima-varshini-/blob/main/front_end_for_screen_rotation_as_orthogonal_transformation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FRONTEND - INSTALL GRADIO
# ============================================================

!pip -q install gradio

In [ ]:
# ============================================================
# FRONTEND IMPORTS
# ============================================================

import gradio as gr
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# FRONTEND BACKEND FUNCTION
# ============================================================

def screen_rotation(angle, x, y, direction):

    # Convert angle to radians
    theta = np.radians(angle)

    # Change sign for clockwise rotation
    if direction == "Clockwise":
        theta = -theta

    # Rotation matrix
    Q = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
    ])

    # Original point
    point = np.array([x, y], dtype=float)

    # Rotated point
    rotated_point = Q @ point

    # Mathematical checks
    QTQ = Q.T @ Q
    determinant = np.linalg.det(Q)

    original_distance = np.linalg.norm(point)
    rotated_distance = np.linalg.norm(rotated_point)

    # Quadratic form with A = I
    A = np.eye(2)

    quadratic_original = point.T @ A @ point
    quadratic_rotated = rotated_point.T @ A @ rotated_point

    return (
        Q,
        rotated_point,
        QTQ,
        determinant,
        original_distance,
        rotated_distance,
        quadratic_original,
        quadratic_rotated
    )

In [ ]:

# ============================================================
# ROTATION GRAPH
# ============================================================

def create_rotation_graph(angle, x, y, direction):

    theta = np.radians(angle)

    if direction == "Clockwise":
        theta = -theta

    Q = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
    ])

    point = np.array([x, y])
    rotated_point = Q @ point

    fig, ax = plt.subplots(figsize=(7, 7))

    # Original vector
    ax.arrow(
        0, 0,
        x, y,
        head_width=max(10, np.linalg.norm(point) * 0.04),
        length_includes_head=True,
        linewidth=2,
        label="Original"
    )

    # Rotated vector
    ax.arrow(
        0, 0,
        rotated_point[0],
        rotated_point[1],
        head_width=max(10, np.linalg.norm(point) * 0.04),
        length_includes_head=True,
        linewidth=2,
        label="Rotated"
    )

    # Points
    ax.scatter(x, y, s=80)
    ax.scatter(
        rotated_point[0],
        rotated_point[1],
        s=80,
        marker="x"
    )

    # Labels
    ax.text(
        x, y,
        f"  Original ({x:.0f}, {y:.0f})"
    )

    ax.text(
        rotated_point[0],
        rotated_point[1],
        f"  Rotated ({rotated_point[0]:.0f}, "
        f"{rotated_point[1]:.0f})"
    )

    # Coordinate axes
    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)

    ax.set_xlabel("X coordinate")
    ax.set_ylabel("Y coordinate")

    ax.set_title(
        f"Screen Coordinate Rotation - {angle}° {direction}"
    )

    ax.grid(True, alpha=0.3)
    ax.set_aspect("equal")
    ax.legend()

    # Automatic limits
    values = np.array([
        [0, 0],
        point,
        rotated_point
    ])

    max_value = max(100, np.max(np.abs(values)) * 1.4)

    ax.set_xlim(-max_value, max_value)
    ax.set_ylim(-max_value, max_value)

    return fig

In [ ]:
# ============================================================
# RESULT DISPLAY FUNCTION
# ============================================================

def calculate_results(angle, x, y, direction):

    (
        Q,
        rotated_point,
        QTQ,
        determinant,
        original_distance,
        rotated_distance,
        quadratic_original,
        quadratic_rotated
    ) = screen_rotation(angle, x, y, direction)

    matrix_text = f"""
Rotation Matrix Q:

[{Q[0,0]:8.4f}  {Q[0,1]:8.4f}]
[{Q[1,0]:8.4f}  {Q[1,1]:8.4f}]
"""

    point_text = f"""
Original point:
({x:.4f}, {y:.4f})

Rotated point:
({rotated_point[0]:.4f}, {rotated_point[1]:.4f})
"""

    orthogonal_text = f"""
QᵀQ:

[{QTQ[0,0]:8.4f}  {QTQ[0,1]:8.4f}]
[{QTQ[1,0]:8.4f}  {QTQ[1,1]:8.4f}]

Orthogonal:
{"PASS ✓" if np.allclose(QTQ, np.eye(2)) else "FAIL ✗"}

det(Q):
{determinant:.6f}

Distance before rotation:
{original_distance:.4f}

Distance after rotation:
{rotated_distance:.4f}

Distance preserved:
{"YES ✓" if np.isclose(original_distance, rotated_distance) else "NO ✗"}
"""

    quadratic_text = f"""
Quadratic Form

q(x) = xᵀAx

Using A = I:

Before rotation:
{quadratic_original:.4f}

After rotation:
{quadratic_rotated:.4f}

Quadratic form invariant:
{"YES ✓" if np.isclose(quadratic_original, quadratic_rotated) else "NO ✗"}
"""

    return (
        matrix_text,
        point_text,
        orthogonal_text,
        quadratic_text
    )

In [ ]:
# ============================================================
# MAIN FRONTEND
# ============================================================

with gr.Blocks(
    title="Screen Rotation as Orthogonal Transformation"
) as app:

    gr.Markdown(
        """
        # 📱 Screen Rotation as Orthogonal Transformation

        ### B.Tech Mathematics Mini Project

        **Explore how a phone screen rotation can be represented
        using an orthogonal rotation matrix.**
        """
    )

    gr.Markdown(
        """
        ### Mathematical Model

        For an angle θ:

        **Q(θ) = [[cosθ, −sinθ], [sinθ, cosθ]]**

        An orthogonal matrix satisfies:

        **QᵀQ = I**

        Therefore, distances and angles are preserved.
        """
    )

    with gr.Row():

        with gr.Column():

            gr.Markdown("## ⚙️ Input")

            angle_input = gr.Slider(
                minimum=0,
                maximum=360,
                value=90,
                step=1,
                label="Rotation Angle (degrees)"
            )

            direction_input = gr.Radio(
                choices=[
                    "Counter-clockwise",
                    "Clockwise"
                ],
                value="Counter-clockwise",
                label="Rotation Direction"
            )

            x_input = gr.Number(
                value=200,
                label="X Coordinate"
            )

            y_input = gr.Number(
                value=300,
                label="Y Coordinate"
            )

            calculate_button = gr.Button(
                "🔄 Rotate Screen",
                variant="primary"
            )

        with gr.Column():

            gr.Markdown("## 📊 Results")

            matrix_output = gr.Textbox(
                label="Rotation Matrix",
                lines=5
            )

            point_output = gr.Textbox(
                label="Coordinate Transformation",
                lines=6
            )

            verification_output = gr.Textbox(
                label="Orthogonality & Distance",
                lines=14
            )

            quadratic_output = gr.Textbox(
                label="Quadratic Form",
                lines=10
            )

    gr.Markdown("## 📈 Visualization")

    graph_output = gr.Plot(
        label="Before and After Rotation"
    )

    calculate_button.click(
        fn=lambda angle, x, y, direction: calculate_results(
            angle,
            x,
            y,
            direction
        ),
        inputs=[
            angle_input,
            x_input,
            y_input,
            direction_input
        ],
        outputs=[
            matrix_output,
            point_output,
            verification_output,
            quadratic_output
        ]
    )

    calculate_button.click(
        fn=lambda angle, x, y, direction: create_rotation_graph(
            angle,
            x,
            y,
            direction
        ),
        inputs=[
            angle_input,
            x_input,
            y_input,
            direction_input
        ],
        outputs=graph_output
    )

    gr.Markdown(
        """
        ---
        ### 🧮 Key Results

        **Orthogonality:** QᵀQ = I

        **Determinant:** det(Q) = 1

        **Distance:** Preserved

        **Quadratic form:** Preserved for A = I

        **Two 90° rotations:** Equivalent to a 180° rotation
        """
    )

In [ ]:
# ============================================================
# LAUNCH FRONTEND
# ============================================================

app.launch(share=True)